## pipeline体验音频应用

In [1]:
import torch

from transformers import pipeline
from datasets import load_dataset, Audio

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### 1. 加载MINDS14数据集

In [3]:
# ds = load_dataset("PolyAI/minds14", name="en-AU", trust_remote_code=True)

ds = load_dataset("PolyAI/minds14", name="en-AU", split="train", trust_remote_code=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

In [4]:
ds

Dataset({
    features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
    num_rows: 654
})

### 2. 体验音频分类

> 使用`Transformers`中的`audio-classification pipeline`来构建一个将音频分入一个类别集的任务。    
> 这里我们使用`anton-l/xtreme_s_xlsr_300m_minds14`模型。

In [5]:
# 模型有点大：有1.26G，第一次下载需要点时间
classifier = pipeline("audio-classification", model="anton-l/xtreme_s_xlsr_300m_minds14")

Some weights of the model checkpoint at anton-l/xtreme_s_xlsr_300m_minds14 were not used when initializing Wav2Vec2ForSequenceClassification: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at anton-l/xtreme_s_xlsr_300m_minds14 and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos

In [6]:
type(ds[0]["audio"]["array"]), ds[0]["audio"]["array"].shape

(numpy.ndarray, (124830,))

In [7]:
classifier(ds[0]["audio"]["array"])

[{'score': 0.9625311493873596, 'label': 'pay_bill'},
 {'score': 0.02867273800075054, 'label': 'freeze'},
 {'score': 0.003349794540554285, 'label': 'card_issues'},
 {'score': 0.0020057999063283205, 'label': 'abroad'},
 {'score': 0.000848432828206569, 'label': 'high_value_payment'}]

根据结果最高概率的是`pay_bill`（96.25%)。

In [8]:
# 查看数据真实的分类
ds.features["intent_class"].int2str(ds[0]["intent_class"])

'pay_bill'

预测正确！

### 3. 自动语音识别

#### 3.1 使用默认的模型

In [9]:
# 不传递模型，默认会使用：facebook/wav2vec2-base-960h
asr = pipeline("automatic-speech-recognition")

No model was supplied, defaulted to facebook/wav2vec2-base-960h and revision 55bb623 (https://hf-mirror.com/facebook/wav2vec2-base-960h).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at facebook/wav2vec2-base-960h were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at faceboo

In [10]:
asr(ds[0]["audio"]["array"])

{'text': 'I WOULD LIKE TO PAY MY ELECTRICITY BILL USING MY CAD CAN YOU PLEASE ASSIST'}

In [11]:
ds[0]["english_transcription"]

'I would like to pay my electricity bill using my card can you please assist'

**哇哦，继续预测正确！**  (不过，card识别为了CAD)。

#### 3.2 使用whisper模型
- https://huggingface.co/openai/whisper-base
- https://huggingface.co/openai/whisper-large-v3

In [12]:
asr2 = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", generate_kwargs={"language": "english"})

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
%time asr2(ds[0]["audio"]["array"])

CPU times: user 30.9 s, sys: 18 s, total: 48.9 s
Wall time: 5.54 s


{'text': ' I would like to pay my electricity bill using my card. Can you please assist?'}

In [14]:
ds[0]["english_transcription"]

'I would like to pay my electricity bill using my card can you please assist'

**非常的准确！**

**whisper-base:**

In [15]:
asr2_base = pipeline("automatic-speech-recognition", model="openai/whisper-base")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [16]:
%time asr2_base(ds[0]["audio"]["array"])

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


CPU times: user 2.72 s, sys: 1.49 s, total: 4.21 s
Wall time: 508 ms


{'text': ' I would like to pay my electricity bill using my card. Can you please assist?'}

使用较小的模型，速度快了很多，也准确。唯一不足是仅仅支持中文的。

**whisper-medium**:

In [17]:
asr2_medium = pipeline("automatic-speech-recognition", model="openai/whisper-medium")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [18]:
%time asr2_medium(ds[0]["audio"]["array"])

CPU times: user 17.9 s, sys: 10.9 s, total: 28.8 s
Wall time: 3.4 s


{'text': ' I would like to pay my electricity bill using my card. Can you please assist?'}

**whisper-small**:

In [19]:
asr2_small = pipeline("automatic-speech-recognition", model="openai/whisper-small")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
%time asr2_small(ds[0]["audio"]["array"])

CPU times: user 6.5 s, sys: 3.54 s, total: 10 s
Wall time: 1.12 s


{'text': ' I would like to pay my electricity bill using my card. Can you please assist?'}